# 1. Загрузка данных

In [1]:
from util.logconf import logging

log = logging.getLogger(__name__)
# log.setLevel(logging.WARN)
# log.setLevel(logging.INFO)
log.setLevel(logging.DEBUG)

### Предвычисление кеша, чтобы CPU не стал узким местом

In [2]:
from precache import LunaPrepCacheApp

LunaPrepCacheApp(['--num-workers=4']).main()

2026-05-06 20:40:17,273 INFO     pid:36582 precache:034:main Starting LunaPrepCacheApp, Namespace(batch_size=1024, num_workers=4)
2026-05-06 20:40:19,302 INFO     pid:36582 dsets:221:__init__ <dsets.LunaDataset object at 0x7ce428e15d30>: 551065 training samples
2026-05-06 20:40:19,710 WARNING  pid:36582 util.util:084:enumerateWithEstimate Stuffing cache ----/539, starting
2026-05-06 20:40:55,698 INFO     pid:36582 util.util:101:enumerateWithEstimate Stuffing cache   16/539, done at 2026-05-06 20:53:45, 0:13:09
2026-05-06 20:42:09,638 INFO     pid:36582 util.util:101:enumerateWithEstimate Stuffing cache   64/539, done at 2026-05-06 20:54:13, 0:13:36
2026-05-06 20:46:53,603 INFO     pid:36582 util.util:101:enumerateWithEstimate Stuffing cache  256/539, done at 2026-05-06 20:53:53, 0:13:17
2026-05-06 20:53:16,916 WARNING  pid:36582 util.util:114:enumerateWithEstimate Stuffing cache ----/539, done at 2026-05-06 20:53:16


# 2. Обучение классификатора на поиск узелков

In [3]:
import torch
from dsets import LunaDataset, CandidateInfoTuple

## 2.1 Точка входа в приложение

In [4]:
def importstr(module_str, from_=None):
    """
    >>> importstr('os')
    <module 'os' from '.../os.pyc'>
    >>> importstr('math', 'fabs')
    <built-in function fabs>
    """
    if from_ is None and ':' in module_str:
        module_str, from_ = module_str.rsplit(':')

    module = __import__(module_str)
    for sub_str in module_str.split('.')[1:]:
        module = getattr(module, sub_str)

    if from_:
        try:
            return getattr(module, from_)
        except:
            raise ImportError('{}.{}'.format(module_str, from_))
    return module

In [5]:
def run(app, *argv):
    argv = list(argv)
    argv.insert(0, '--num-workers=4')
    log.info(f'Running: {app}({argv}).main()')

    if isinstance(app, str):
        app_cls = importstr(*app.rsplit('.', 1))        
    else:
        app_cls = app
        
    app_cls(argv).main()

    log.info(f'Finished: {app}({argv}).main()')

In [19]:
import argparse
import sys
import datetime
from model import LunaModel
from torch.optim import SGD
import numpy as np
from torch.utils.data import DataLoader
from util.util import enumerateWithEstimate
import torch.nn as nn

METRICS_LABEL_NDX=0
METRICS_PRED_NDX=1
METRICS_LOSS_NDX=2
METRICS_SIZE = 3

class LunaTrainingApp:
    def __init__(self, sys_argv=None):
        if sys_argv is None:
            sys_argv = sys.argv[1:]

        parser = argparse.ArgumentParser()
        parser.add_argument('--num-workers', help='Number of worker process for background data loading', default=8, type=int)
        parser.add_argument('--batch-size', help='Batch size to use for training', default=64, type=int)
        parser.add_argument('--epochs', help='Number of epochs to train for', default=1, type=int)

        self.cli_args = parser.parse_args(sys_argv)
        self.time_str = datetime.datetime.now().strftime('%Y-%m-%d_%H.%M.%S')

        self.use_cuda = torch.cuda.is_available()
        self.device = torch.device("cuda" if self.use_cuda else "cpu")

        self.model = self.initModel()
        self.optimizer = self.initOptimizer()

        self.totalTrainingSamplesCount = 0

    def main(self):
        log.info(f'Starting {type(self).__name__}, {self.cli_args}')
        train_dl = self.initTrainDl()
        val_dl = self.initValDl()

        for epoch_ndx in range(1, self.cli_args.epochs + 1):
            log.info("Epoch {} of {}, {}/{} batches of size {}*{}".format(
                epoch_ndx,
                self.cli_args.epochs,
                len(train_dl),
                len(val_dl),
                self.cli_args.batch_size,
                (torch.cuda.device_count() if self.use_cuda else 1),
            ))

            trnMetrics_t = self.doTraining(epoch_ndx, train_dl)
            self.logMetrics(epoch_ndx, 'trn', trnMetrics_t)

            valMetrics_t = self.doValidation(epoch_ndx, val_dl)
            self.logMetrics(epoch_ndx, 'val', valMetrics_t)


    def initModel(self):
        model = LunaModel()
        if self.use_cuda:
            log.info(f'Using CUDA; {torch.cuda.device_count()} devices.')
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(self.device)
        return model

    def doTraining(self, epoch_ndx, train_dl):
        self.model.train()
        trnMetrics_g = torch.zeros( # пустой массив для метрик
            METRICS_SIZE,
            len(train_dl.dataset),
            device=self.device,
        )
        batch_iter = enumerateWithEstimate( # измерение предполагаемого времени завершения
            train_dl,
            "E{} Training".format(epoch_ndx),
            start_ndx=train_dl.num_workers,
        )
        for batch_ndx, batch_tup in batch_iter:
            self.optimizer.zero_grad()
            loss_var = self.computeBatchLoss(
                batch_ndx,
                batch_tup,
                train_dl.batch_size,
                trnMetrics_g
            )

            # обновление весовых коэффициентов
            loss_var.backward() 
            self.optimizer.step()

        self.totalTrainingSamplesCount += len(train_dl.dataset)

        return trnMetrics_g.to('cpu')

    def doValidation(self, epoch_ndx, val_dl):
        with torch.no_grad():
            self.model.eval() # отключение логики обучения
            valMetrics_g = torch.zeros(
                METRICS_SIZE,
                len(val_dl.dataset),
                device=self.device,
            )

            batch_iter = enumerateWithEstimate(
                val_dl,
                "E{} Validation ".format(epoch_ndx),
                start_ndx=val_dl.num_workers,
            )
            for batch_ndx, batch_tup in batch_iter:
                self.computeBatchLoss(batch_ndx, batch_tup, val_dl.batch_size, valMetrics_g)

        return valMetrics_g.to('cpu')

    def computeBatchLoss(self, batch_ndx, batch_tup, batch_size, metrics_g):
        '''Вычисление потерь для каждой точки данных (не усреднённое по пакету)'''
        input_t, label_t, _series_list, _center_list = batch_tup

        input_g = input_t.to(self.device, non_blocking=True)
        label_g = label_t.to(self.device, non_blocking=True)

        logits_g, probability_g = self.model(input_g)

        loss_func = nn.CrossEntropyLoss(reduction='none') # потери для точки данных
        loss_g = loss_func(
            logits_g,
            label_g[:,1], # индекс класса
        )
        start_ndx = batch_ndx * batch_size
        end_ndx = start_ndx + label_t.size(0)

        # detach, чтобы метрики не удерживали градиенты
        metrics_g[METRICS_LABEL_NDX, start_ndx:end_ndx] = \
            label_g[:,1].detach()
        metrics_g[METRICS_PRED_NDX, start_ndx:end_ndx] = \
            probability_g[:,1].detach()
        metrics_g[METRICS_LOSS_NDX, start_ndx:end_ndx] = \
            loss_g.detach()

        return loss_g.mean() # среднее потерь точки в одно значение

    def initOptimizer(self):
        return SGD(self.model.parameters(), lr=0.001, momentum=0.99) # Adam(self.model.parameters())

    def _initDl(self, is_val):
        ds = LunaDataset(
            val_stride=10,
            isValSet_bool=is_val,
        )
    
        batch_size = self.cli_args.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()
    
        return DataLoader(
            ds,
            batch_size=batch_size,
            num_workers=self.cli_args.num_workers,
            pin_memory=self.use_cuda,
        )


    def initTrainDl(self):
        return self._initDl(False)
    
    
    def initValDl(self):
        return self._initDl(True)

    def logMetrics(self, epoch_ndx, mode_str, metrics_t, classificationThreshold=0.5):
        '''Логирование метрик. mode_str - метрики предназначены для обучения или проверки'''
        
        # составление маски для определения метрик только с узелками (/pos) или без узелков (/neg)
        negLabel_mask = metrics_t[METRICS_LABEL_NDX] <= classificationThreshold
        negPred_mask = metrics_t[METRICS_PRED_NDX] <= classificationThreshold
        posLabel_mask = ~negLabel_mask
        posPred_mask = ~negPred_mask

        # вычисление статистических данных для метрик
        neg_count = int(negLabel_mask.sum())
        pos_count = int(posLabel_mask.sum())

        neg_correct = int((negLabel_mask & negPred_mask).sum())
        pos_correct = int((posLabel_mask & posPred_mask).sum())

        metrics_dict = {}
        metrics_dict['loss/all'] = metrics_t[METRICS_LOSS_NDX].mean()
        metrics_dict['loss/neg'] = metrics_t[METRICS_LOSS_NDX, negLabel_mask].mean()
        metrics_dict['loss/pos'] = metrics_t[METRICS_LOSS_NDX, posLabel_mask].mean()

        metrics_dict['correct/all'] = (pos_correct + neg_correct) / np.float32(metrics_t.shape[1]) * 100
        metrics_dict['correct/neg'] = neg_correct / np.float32(neg_count) * 100
        metrics_dict['correct/pos'] = pos_correct / np.float32(pos_count) * 100

        log.info(
            ("E{} {:8} {loss/all:.4f} loss, "
                 + "{correct/all:-5.1f}% correct, "
            ).format(
                epoch_ndx,
                mode_str,
                **metrics_dict,
            )
        )
        log.info(
            ("E{} {:8} {loss/neg:.4f} loss, "
                 + "{correct/neg:-5.1f}% correct ({neg_correct:} of {neg_count:})"
            ).format(
                epoch_ndx,
                mode_str + '_neg',
                neg_correct=neg_correct,
                neg_count=neg_count,
                **metrics_dict,
            )
        )
        log.info(
            ("E{} {:8} {loss/pos:.4f} loss, "
                 + "{correct/pos:-5.1f}% correct ({pos_correct:} of {pos_count:})"
            ).format(
                epoch_ndx,
                mode_str + '_pos',
                pos_correct=pos_correct,
                pos_count=pos_count,
                **metrics_dict,
            )
        )

#if __name__ == '__main__':
#    LunaTrainingApp().main()

In [20]:
run(LunaTrainingApp, '--epochs=1')

2026-05-07 00:02:56,836 INFO     pid:36582 __main__:004:run Running: <class '__main__.LunaTrainingApp'>(['--num-workers=4', '--epochs=1']).main()
2026-05-07 00:02:56,841 INFO     pid:36582 __main__:062:initModel Using CUDA; 1 devices.
2026-05-07 00:02:56,860 INFO     pid:36582 __main__:038:main Starting LunaTrainingApp, Namespace(num_workers=4, batch_size=64, epochs=1)
2026-05-07 00:02:56,878 INFO     pid:36582 dsets:221:__init__ <dsets.LunaDataset object at 0x7ce400fc4ad0>: 495958 training samples
2026-05-07 00:02:56,902 INFO     pid:36582 dsets:221:__init__ <dsets.LunaDataset object at 0x7ce4116d9640>: 55107 validation samples
2026-05-07 00:02:56,904 INFO     pid:36582 __main__:043:main Epoch 1 of 1, 7750/862 batches of size 64*1
2026-05-07 00:02:56,906 WARNING  pid:36582 util.util:084:enumerateWithEstimate E1 Training ----/7750, starting
2026-05-07 00:03:13,299 INFO     pid:36582 util.util:101:enumerateWithEstimate E1 Training   16/7750, done at 2026-05-07 00:35:16, 0:32:06
2026-05-